# Proposition Chunking

## Overview

Instead of retrieving large chunks of text, **proposition chunking** breaks a document into small, atomic, self-contained facts — then retrieves those precise propositions.

| Standard Chunking | Proposition Chunking |
|---|---|
| Splits text by character count | Splits text into **individual facts** |
| One chunk = many mixed ideas | One proposition = one specific fact |
| May return irrelevant context | Returns only the relevant fact |

Based on [research from Tony Chen, et al.](https://doi.org/10.48550/arXiv.2312.06648)

## Pipeline

1. Split document into manageable chunks
2. Use LLM to generate **propositions** from each chunk (atomic, factual, self-contained statements)
3. **Quality check** each proposition (accuracy, clarity, completeness, conciseness)
4. Embed propositions into a vector store for retrieval
5. Compare retrieval vs. standard larger-chunk retrieval

<img src="./images/proposition_chunking.svg" alt="Proposition Chunking" width="600">

## Models Used

- **LLM**: `gemma3:4b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

---
## Step 0: Import Packages

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

---
## Step 1: Set Up LLM and Embedding Model

In [2]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")
llm = ChatOllama(model="gemma3:4b", temperature=0)

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Define the Sample Document

We use a passage about Paul Graham's "Founder Mode" essay as our test document.

In [3]:
sample_content = """Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.
Conventional Wisdom vs. Founder Mode
The essay argues that the traditional advice given to growing companies\u2014hiring good people and giving them autonomy\u2014often fails when applied to startups.
This approach, suitable for established companies, can be detrimental to startups where the founder's vision and direct involvement are crucial. \"Founder Mode\" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional \"manager mode\" often advised by business schools and professional managers.
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's vision and culture.
Graham suggests that founders should leverage these strengths rather than conform to traditional managerial practices. \"Founder Mode\" is an emerging paradigm that is not yet fully understood or documented, with Graham hoping that over time, it will become as well-understood as the traditional manager mode, allowing founders to maintain their unique approach even as their companies scale.
Challenges of Scaling Startups
As startups grow, there is a common belief that they must transition to a more structured managerial approach. However, many founders have found this transition problematic, as it often leads to a loss of the innovative and agile spirit that drove the startup's initial success.
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's \"Founder Mode\" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart
. This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale.
"""

print(f"Document length: {len(sample_content)} characters")

Document length: 2649 characters


---
## Step 3: Split Document into Chunks

We split the text into small chunks (200 tokens each with 50 overlap). These chunks will be the input for proposition generation.

In [4]:
docs_list = [Document(
    page_content=sample_content,
    metadata={
        "Title": "Paul Graham's Founder Mode Essay",
        "Source": "https://www.perplexity.ai/page/paul-graham-s-founder-mode-ess-t9TCyvkqRiyMQJWsHr0fnQ"
    }
)]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)

for i, doc in enumerate(doc_splits):
    doc.metadata["chunk_id"] = i + 1

print(f"Split into {len(doc_splits)} chunks:")
for i, doc in enumerate(doc_splits):
    print(f"  Chunk {i+1}: {doc.page_content[:80]}...")

Split into 3 chunks:
  Chunk 1: Paul Graham's essay "Founder Mode," published in September 2024, challenges conv...
  Chunk 2: Unique Founder Abilities
Founders possess unique insights and abilities that pro...
  Chunk 3: Brian Chesky, co-founder of Airbnb, shared his experience of being advised to ru...


---
## Step 4: Generate Propositions from Each Chunk

For each chunk, we ask the LLM to break it into **atomic, self-contained propositions** — individual facts that can be understood on their own.

We use **few-shot prompting** with an example (Neil Armstrong) to show the LLM what good propositions look like.

In [5]:
proposition_schema = {
    "title": "GeneratePropositions",
    "description": "List of all the propositions in a given document",
    "type": "object",
    "properties": {
        "propositions": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of propositions (factual, self-contained, and concise)"
        }
    },
    "required": ["propositions"]
}

# Few-shot example
proposition_examples = [
    {
        "document": "In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.",
        "propositions": (
            "['Neil Armstrong was an astronaut.', "
            "'Neil Armstrong walked on the Moon in 1969.', "
            "'Neil Armstrong was the first person to walk on the Moon.', "
            "'Neil Armstrong walked on the Moon during the Apollo 11 mission.', "
            "'The Apollo 11 mission occurred in 1969.']"
        )
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{document}"),
    ("ai", "{propositions}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=proposition_examples,
)

system_msg = (
    "Please break down the following text into simple, self-contained propositions. "
    "Ensure that each proposition meets the following criteria:\n\n"
    "1. Express a Single Fact: Each proposition should state one specific fact or claim.\n"
    "2. Be Understandable Without Context: Self-contained, no additional context needed.\n"
    "3. Use Full Names, Not Pronouns: Avoid pronouns; use full entity names.\n"
    "4. Include Relevant Dates/Qualifiers: Include dates, times, qualifiers when applicable.\n"
    "5. Contain One Subject-Predicate Relationship: One subject and its action/attribute."
)

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    few_shot_prompt,
    ("human", "{document}"),
])

proposition_chain = gen_prompt | llm.with_structured_output(proposition_schema)

print("Proposition generation chain ready")

Proposition generation chain ready


In [6]:
propositions = []

for i, chunk in enumerate(doc_splits):
    result = proposition_chain.invoke({"document": chunk.page_content})
    chunk_propositions = result["propositions"]
    print(f"Chunk {i+1}: generated {len(chunk_propositions)} propositions")

    for prop_text in chunk_propositions:
        propositions.append(Document(
            page_content=prop_text,
            metadata={
                "Title": "Paul Graham's Founder Mode Essay",
                "Source": "https://www.perplexity.ai/page/paul-graham-s-founder-mode-ess-t9TCyvkqRiyMQJWsHr0fnQ",
                "chunk_id": i + 1
            }
        ))

print(f"\nTotal propositions generated: {len(propositions)}")
print("\nSample propositions:")
for p in propositions[:5]:
    print(f"  - {p.page_content}")

Chunk 1: generated 8 propositions
Chunk 2: generated 10 propositions
Chunk 3: generated 11 propositions

Total propositions generated: 29

Sample propositions:
  - Paul Graham published an essay titled ‘Founder Mode’ in September 2024.
  - The ‘Founder Mode’ essay challenges conventional wisdom about scaling startups.
  - Conventional wisdom suggests hiring good people and giving them autonomy for growing companies.
  - The ‘Founder Mode’ approach argues that founder’s direct involvement is crucial for startups.
  - ‘Founder Mode’ is presented as an emerging paradigm.


---
## Step 5: Quality Check Each Proposition

Not all generated propositions are good. We score each one on 4 criteria (1-10 scale):

| Criterion | What it measures |
|---|---|
| **Accuracy** | Does it correctly reflect the original text? |
| **Clarity** | Can it be understood without additional context? |
| **Completeness** | Does it include necessary dates/qualifiers? |
| **Conciseness** | Is it brief without losing important information? |

Propositions must score **>= 7 on all 4 criteria** to pass.

In [7]:
grade_schema = {
    "title": "GradePropositions",
    "description": "Grade a proposition on accuracy, clarity, completeness, and conciseness",
    "type": "object",
    "properties": {
        "accuracy":     {"type": "integer", "description": "Rate 1-10: how well the proposition reflects the original text"},
        "clarity":      {"type": "integer", "description": "Rate 1-10: how easy it is to understand without additional context"},
        "completeness": {"type": "integer", "description": "Rate 1-10: whether it includes necessary details (dates, qualifiers)"},
        "conciseness":  {"type": "integer", "description": "Rate 1-10: whether it is concise without losing important information"}
    },
    "required": ["accuracy", "clarity", "completeness", "conciseness"]
}

eval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Please evaluate the following proposition based on these criteria:\n"
     "- **Accuracy**: Rate 1-10 based on how well the proposition reflects the original text.\n"
     "- **Clarity**: Rate 1-10 based on how easy it is to understand without additional context.\n"
     "- **Completeness**: Rate 1-10 based on whether it includes necessary details (dates, qualifiers).\n"
     "- **Conciseness**: Rate 1-10 based on whether it is concise without losing important information.\n\n"
     "Example:\n"
     "Docs: In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.\n"
     "Proposition: Neil Armstrong walked on the Moon in 1969.\n"
     "Evaluation: accuracy=10, clarity=10, completeness=10, conciseness=10"),
    ("human",
     "Proposition: \"{proposition}\"\nOriginal Text: \"{original_text}\""),
])

evaluator_chain = eval_prompt | llm.with_structured_output(grade_schema)

print("Proposition evaluator chain ready")

Proposition evaluator chain ready


In [8]:
thresholds = {"accuracy": 7, "clarity": 7, "completeness": 7, "conciseness": 7}

evaluated_propositions = []

for idx, prop in enumerate(propositions):
    original_text = doc_splits[prop.metadata["chunk_id"] - 1].page_content
    scores = evaluator_chain.invoke({"proposition": prop.page_content, "original_text": original_text})

    passed = all(scores[cat] >= thresholds[cat] for cat in thresholds)

    if passed:
        evaluated_propositions.append(prop)
    else:
        print(f"FAILED #{idx+1}: \"{prop.page_content}\"")
        print(f"  Scores: accuracy={scores['accuracy']}, clarity={scores['clarity']}, "
              f"completeness={scores['completeness']}, conciseness={scores['conciseness']}")

print(f"\nPassed: {len(evaluated_propositions)} / {len(propositions)} propositions")

FAILED #19: "Steve Jobs was a manager."
  Scores: accuracy=6, clarity=9, completeness=7, conciseness=8

Passed: 28 / 29 propositions


---
## Step 6: Build Two Vector Stores for Comparison

We create two separate FAISS vector stores:
1. **Proposition-based** — embeddings of individual propositions (fine-grained)
2. **Chunk-based** — embeddings of the original larger chunks (coarse-grained)

In [9]:
vectorstore_propositions = FAISS.from_documents(evaluated_propositions, embedding_model)
retriever_propositions = vectorstore_propositions.as_retriever(
    search_type="similarity", search_kwargs={"k": 4}
)
print(f"Proposition vector store: {len(evaluated_propositions)} propositions")

vectorstore_larger = FAISS.from_documents(doc_splits, embedding_model)
retriever_larger = vectorstore_larger.as_retriever(
    search_type="similarity", search_kwargs={"k": 4}
)
print(f"Chunk vector store: {len(doc_splits)} chunks")

Proposition vector store: 28 propositions
Chunk vector store: 3 chunks


---
## Step 7: Test Retrieval — Compare Proposition vs. Chunk

We run several test queries on both retrievers to see how proposition-based retrieval compares to standard chunk-based retrieval.

---
### Query 1: Who inspired Brian Chesky's Founder Mode?

In [10]:
query = "Who's management approach served as inspiration for Brian Chesky's \"Founder Mode\" at Airbnb?"
print(f"Query: {query}\n")

print("--- Proposition retrieval ---")
res_prop = retriever_propositions.invoke(query)
for i, r in enumerate(res_prop):
    print(f"  {i+1}) {r.page_content}  [chunk {r.metadata['chunk_id']}]")

print("\n--- Chunk retrieval ---")
res_chunk = retriever_larger.invoke(query)
for i, r in enumerate(res_chunk):
    print(f"  {i+1}) {r.page_content[:120]}...  [chunk {r.metadata['chunk_id']}]")

Query: Who's management approach served as inspiration for Brian Chesky's "Founder Mode" at Airbnb?

--- Proposition retrieval ---
  1) ‘Founder Mode’ is an emerging paradigm.  [chunk 2]
  2) ‘Founder Mode’ is presented as an emerging paradigm.  [chunk 1]
  3) Founders possess unique insights.  [chunk 2]
  4) The ‘Founder Mode’ approach argues that founder’s direct involvement is crucial for startups.  [chunk 1]

--- Chunk retrieval ---
  1) Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional manageria...  [chunk 3]
  2) Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, ...  [chunk 1]
  3) Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily bec...  [chunk 2]


---
### Query 2: What is the essay "Founder Mode" about?

In [11]:
query2 = 'what is the essay "Founder Mode" about?'
print(f"Query: {query2}\n")

print("--- Proposition retrieval ---")
res_prop2 = retriever_propositions.invoke(query2)
for i, r in enumerate(res_prop2):
    print(f"  {i+1}) {r.page_content}  [chunk {r.metadata['chunk_id']}]")

print("\n--- Chunk retrieval ---")
res_chunk2 = retriever_larger.invoke(query2)
for i, r in enumerate(res_chunk2):
    print(f"  {i+1}) {r.page_content[:120]}...  [chunk {r.metadata['chunk_id']}]")

Query: what is the essay "Founder Mode" about?

--- Proposition retrieval ---
  1) The ‘Founder Mode’ essay challenges conventional wisdom about scaling startups.  [chunk 1]
  2) ‘Founder Mode’ is presented as an emerging paradigm.  [chunk 1]
  3) ‘Founder Mode’ is an emerging paradigm.  [chunk 2]
  4) ‘Founder Mode’ is not yet fully understood.  [chunk 2]

--- Chunk retrieval ---
  1) Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, ...  [chunk 1]
  2) Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily bec...  [chunk 2]
  3) Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional manageria...  [chunk 3]


---
### Query 3: Who is the co-founder of Airbnb?

In [12]:
query3 = "who is the co-founder of Airbnb?"
print(f"Query: {query3}\n")

print("--- Proposition retrieval ---")
res_prop3 = retriever_propositions.invoke(query3)
for i, r in enumerate(res_prop3):
    print(f"  {i+1}) {r.page_content}  [chunk {r.metadata['chunk_id']}]")

print("\n--- Chunk retrieval ---")
res_chunk3 = retriever_larger.invoke(query3)
for i, r in enumerate(res_chunk3):
    print(f"  {i+1}) {r.page_content[:120]}...  [chunk {r.metadata['chunk_id']}]")

Query: who is the co-founder of Airbnb?

--- Proposition retrieval ---
  1) The retreat emphasized founder involvement.  [chunk 3]
  2) Founders possess unique insights.  [chunk 2]
  3) ‘Founder Mode’ is an emerging paradigm.  [chunk 2]
  4) ‘Founder Mode’ is presented as an emerging paradigm.  [chunk 1]

--- Chunk retrieval ---
  1) Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional manageria...  [chunk 3]
  2) Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, ...  [chunk 1]
  3) Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily bec...  [chunk 2]


---
### Query 4: When was the essay published?

In [13]:
query4 = 'when was the essay "founder mode" published?'
print(f"Query: {query4}\n")

print("--- Proposition retrieval ---")
res_prop4 = retriever_propositions.invoke(query4)
for i, r in enumerate(res_prop4):
    print(f"  {i+1}) {r.page_content}  [chunk {r.metadata['chunk_id']}]")

print("\n--- Chunk retrieval ---")
res_chunk4 = retriever_larger.invoke(query4)
for i, r in enumerate(res_chunk4):
    print(f"  {i+1}) {r.page_content[:120]}...  [chunk {r.metadata['chunk_id']}]")

Query: when was the essay "founder mode" published?

--- Proposition retrieval ---
  1) ‘Founder Mode’ is presented as an emerging paradigm.  [chunk 1]
  2) ‘Founder Mode’ is an emerging paradigm.  [chunk 2]
  3) The ‘Founder Mode’ essay challenges conventional wisdom about scaling startups.  [chunk 1]
  4) Paul Graham published an essay titled ‘Founder Mode’ in September 2024.  [chunk 1]

--- Chunk retrieval ---
  1) Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, ...  [chunk 1]
  2) Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily bec...  [chunk 2]
  3) Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional manageria...  [chunk 3]


---
## Comparison Summary

| Aspect | Proposition-Based | Standard Chunk-Based |
|---|---|---|
| **Precision** | High — returns focused, direct facts | Medium — may include irrelevant context |
| **Clarity** | High — each result is one clear fact | Medium — results are long paragraphs |
| **Context** | Low — may lack surrounding context | High — preserves full narrative flow |
| **Best for** | Quick, specific factual queries | Complex queries needing in-depth understanding |
| **Information overload** | Low — concise results | High — more text to sift through |

**Key insight:** Proposition chunking trades context for precision. For factual questions ("Who?", "When?", "What?"), propositions return exactly the right fact. For broader questions requiring understanding of relationships, larger chunks provide richer context.